# Homework 4: Avaliação de modelos (lead scoring)
ML Zoomcamp 2026

[Versão original no Google Colab](https://colab.research.google.com/drive/1oxsqVYrW4VpeWSTd8sk1NPWVrn2JLq0E?usp=sharing)

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('course_lead_scoring_2026.csv')

categorical = list(df.dtypes[df.dtypes == 'object'].index)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.converted.values
y_val = df_val.converted.values

df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)

print(len(df_train), len(df_val), len(df_test))

## Q1: AUC por variável numérica (treino)
**Resposta:** `lead_score` (0.7882)

In [ ]:
for c in ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']:
    auc = roc_auc_score(y_train, df_train[c])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[c])
    print(c, round(auc, 4))

## Q2: AUC do modelo na validação
**Resposta:** 0.732

In [ ]:
features = categorical + numerical

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
print('Q2:', round(roc_auc_score(y_val, y_pred), 3))

## Q3 e Q4: precision, recall e F1
**Q3:** 0.63 | **Q4:** 0.41 (F1 = 0.7576)

In [ ]:
thresholds = np.round(np.arange(0.0, 1.01, 0.01), 2)

rows = []
for t in thresholds:
    pred = (y_pred >= t)
    tp = ((pred == 1) & (y_val == 1)).sum()
    fp = ((pred == 1) & (y_val == 0)).sum()
    fn = ((pred == 0) & (y_val == 1)).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    rows.append((t, p, r, f1))

scores = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'f1'])

plt.plot(scores.threshold, scores.precision, label='precision')
plt.plot(scores.threshold, scores.recall, label='recall')
plt.xlabel('threshold')
plt.legend()
plt.show()

valid = scores[(scores.precision > 0) | (scores.recall > 0)].copy()
valid['diff'] = (valid.precision - valid.recall).abs()
print('Q3:', valid.loc[valid['diff'].idxmin(), 'threshold'])
print('Q4:', scores.loc[scores.f1.idxmax(), 'threshold'], '| F1 =', round(scores.f1.max(), 4))

## Q5: validação cruzada com 5 folds
**Resposta:** std = 0.007

In [ ]:
def train_cv(df_tr, y_tr, C=1.0):
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(df_tr[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X, y_tr)
    return dv, model

def predict_cv(df_v, dv, model):
    X = dv.transform(df_v[features].to_dict(orient='records'))
    return model.predict_proba(X)[:, 1]

df_fold = df_full_train.reset_index(drop=True)
kfold = KFold(n_splits=5, shuffle=True, random_state=1)

aucs = []
for tr_idx, va_idx in kfold.split(df_fold):
    d_tr = df_fold.iloc[tr_idx]
    d_va = df_fold.iloc[va_idx]
    y_tr = d_tr.converted.values
    y_va = d_va.converted.values

    dv_f, m_f = train_cv(d_tr, y_tr)
    aucs.append(roc_auc_score(y_va, predict_cv(d_va, dv_f, m_f)))

print(np.round(aucs, 4))
print('Q5 std:', round(np.std(aucs), 3))

## Q6: afinar C com validação cruzada
**Resposta:** C = 0.001 (média 0.749)

In [ ]:
for C in [0.000001, 0.001, 1]:
    aucs = []
    for tr_idx, va_idx in kfold.split(df_fold):
        d_tr = df_fold.iloc[tr_idx]
        d_va = df_fold.iloc[va_idx]
        y_tr = d_tr.converted.values
        y_va = d_va.converted.values

        dv_f, m_f = train_cv(d_tr, y_tr, C=C)
        aucs.append(roc_auc_score(y_va, predict_cv(d_va, dv_f, m_f)))

    print(C, '| mean:', round(np.mean(aucs), 3), '| std:', round(np.std(aucs), 3))